In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from transformers import LxmertForPreTraining, LxmertTokenizer, LxmertImageProcessor
from tqdm import tqdm

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'LxmertImageProcessor' from 'transformers' (/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/transformers/__init__.py)

In [ ]:

# -- 3. 데이터 로드 --
df = pd.read_csv(TEST_CSV_PATH)
print(f"Test data loaded. Total questions: {len(df)}")

# -- 4. 추론 실행 --
results = []
# tqdm을 사용하여 진행 상황을 시각적으로 표시합니다.
for idx, row in tqdm(df.iterrows(), total=len(df)):
    
    # 이미지 경로 조합 및 이미지 로드
    img_path = os.path.join(IMAGE_BASE_PATH, row['img_path'].lstrip('./'))
    try:
        # LXMERT는 일반적인 RGB 이미지 리스트를 입력으로 받습니다.
        image = Image.open(img_path).convert("RGB")
    except FileNotFoundError:
        print(f"Error: Image not found at {img_path}")
        results.append({'ID': row['ID'], 'answer': -1}) # 에러 발생 시 -1 기록
        continue

    # 질문과 선택지
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]
    
    choice_scores = []
    
    # 4개의 선택지에 대해 각각 관련성 점수 계산
    with torch.no_grad():
        for choice in choices:
            # 텍스트 전처리: 질문과 선택지를 함께 토크나이징
            text_inputs = tokenizer(
                text=question,
                text_pair=choice,
                return_tensors="pt",
                padding="max_length",
                max_length=40, # 적절한 max_length 설정
                truncation=True
            )
            
            # 이미지 전처리
            image_inputs = image_processor(
                images=image,
                return_tensors="pt"
            )

            # 모든 입력을 DEVICE로 이동
            inputs = {k: v.to(DEVICE) for k, v in {**text_inputs, **image_inputs}.items()}

            # 모델을 통해 관련성 점수(cross_relationship_score) 획득
            outputs = model(**inputs)
            
            # outputs.cross_relationship_score는 [unrelated_logit, related_logit] 형태
            # 우리는 '관련 있음(related)'의 logit 점수를 사용
            score = outputs.cross_relationship_score[0][1].item()
            choice_scores.append(score)

    # 점수가 가장 높은 선택지를 정답으로 선택 (1-based index)
    # choice_scores.index(max(choice_scores))는 0, 1, 2, 3 중 하나를 반환
    predicted_answer = choice_scores.index(max(choice_scores)) + 1
    results.append({'ID': row['ID'], 'answer': predicted_answer})

# -- 5. 제출 파일 생성 --
submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

print(f"Inference complete. Submission file saved to {SUBMISSION_CSV_PATH}")
print("\n--- Submission Preview ---")
print(submission_df.head())
print("------------------------")